# Flash Texter — 단계 1 학습 (Colab)

의도 분류 기반 챗봇을 학습합니다. Cloud Press 리포지토리: https://github.com/choichoi3227-crypto/cloud-press

**설계 노트**: 순수 seq2seq 생성 대신, 인코더(임베딩+GRU)가 실제로 학습되어 입력의 의도를 분류하고, 검증된 응답 후보 중 하나를 반환합니다. 자세한 이유는 `docs/03-flash-texter-spec.md`를 참고하세요.

## 1. 리포지토리 클론

In [ ]:
!git clone https://github.com/choichoi3227-crypto/cloud-press.git
%cd cloud-press/ai-models/training/flash-texter
!pip install -q torch numpy

## 2. GPU 확인 (이 모델은 매우 작아서 CPU로도 충분히 빠르지만, 확인해둡니다)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

## 3. Google Drive 마운트 (체크포인트 저장용)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/cloud-press/checkpoints/flash-texter"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("체크포인트 저장 위치:", CHECKPOINT_DIR)

## 4. 시드 대화 데이터 생성
실전에서는 이 셀 대신 AI Hub 등에서 받은 실제 대화 데이터셋(JSONL, {intent, input, response} 형식)을 사용하는 것을 권장합니다 (`docs/02-training-pipeline.md` 5절 참조).

In [ ]:
DATA_PATH = "/content/drive/MyDrive/cloud-press/data/flash_texter_dialogues.jsonl"
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
!python generate_seed_data.py --out {DATA_PATH} --repeat 60

## 5. 학습 실행
체크포인트가 있으면 자동으로 이어서 학습합니다.

In [ ]:
!python train.py --data {DATA_PATH} --checkpoint-dir {CHECKPOINT_DIR} --epochs 50 --batch-size 16

## 6. 직접 대화해보기 (정성 평가)

In [ ]:
import sys
sys.path.insert(0, '.')
from inference import load_model, generate_response

load_model(f"{CHECKPOINT_DIR}/flash_texter_final.pt")

test_inputs = [
    "안녕하세요",
    "오늘 날씨 어때?",
    "고마워요",
    "너는 누구야?",
    "잘 지내?",
    "이만 갈게",
    "완전 관련없는 아무 말",
]
for t in test_inputs:
    print(f"입력: {t}")
    print(f"응답: {generate_response(t)}")
    print()

## 7. Hugging Face Hub에 업로드

In [ ]:
from huggingface_hub import login, HfApi

login()

HF_REPO_ID = "<your-username>/flash-texter"  # 실제 사용자명으로 변경
MODEL_VERSION = "v0.1.0"

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
api.upload_file(
    path_or_fileobj=f"{CHECKPOINT_DIR}/flash_texter_final.pt",
    path_in_repo=f"flash-texter-{MODEL_VERSION}.pt",
    repo_id=HF_REPO_ID,
)
print("업로드 완료:", HF_REPO_ID)